# Credit Card Fraud Detection

## Objective
Build a machine learning pipeline to detect fraudulent financial transactions from a heavily imbalanced dataset, with class imbalance as the core challenge.

**Dataset:** Credit Card Fraud Detection benchmark dataset  
**Rows:** 284,807  
**Fraud cases:** 492  
**Fraud rate:** 0.1727%

**Tech Stack:** Python, pandas, NumPy, scikit-learn, imbalanced-learn (SMOTE), matplotlib, seaborn, Jupyter Notebook

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, roc_curve, classification_report
from imblearn.over_sampling import SMOTE

df = pd.read_csv("creditcard.csv")
df.head()

## 1. Class Imbalance Analysis

In [ ]:
print("Dataset shape:", df.shape)
print("\nClass counts:")
print(df["Class"].value_counts())

fraud_rate = df["Class"].mean() * 100
print(f"\nFraud percentage: {fraud_rate:.4f}%")
print(f"Non-fraud percentage: {100-fraud_rate:.4f}%")
print("Missing values:", df.isnull().sum().sum())

### Why standard accuracy is misleading

Fraud represents only about **0.17%** of all transactions. A model that predicts every transaction as legitimate could achieve about **99.83% accuracy** while detecting no fraud at all.

Therefore, accuracy is not an appropriate primary metric. We use **Precision, Recall, F1-score and ROC-AUC**.

## 2. Exploratory Data Analysis

In [ ]:
# Transaction amount distribution
plt.figure(figsize=(9,5))
sns.histplot(data=df, x="Amount", hue="Class", bins=60, element="step", stat="density", common_norm=False)
plt.xlim(0, df["Amount"].quantile(0.99))
plt.title("Transaction Amount Distribution: Fraud vs Genuine")
plt.show()

# Time-of-day analysis (Time is seconds elapsed from the first transaction)
df["Hour"] = (df["Time"] / 3600) % 24
plt.figure(figsize=(10,5))
sns.histplot(data=df, x="Hour", hue="Class", bins=24, element="step", stat="density", common_norm=False)
plt.title("Transaction Activity by Time of Day")
plt.xlabel("Approximate Hour")
plt.show()

## 3. Train/Test Split and SMOTE

A **stratified 80/20 split** is used so fraud cases appear in both sets.

SMOTE is applied **only to the training set** to prevent data leakage. A sampling ratio of 0.10 is used instead of fully balancing the huge dataset, making the pipeline more practical while still giving the minority class substantially more representation.

In [ ]:
X = df.drop(columns=["Class", "Hour"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Standardize Amount using training data only
scaler = StandardScaler()
X_train[["Amount"]] = scaler.fit_transform(X_train[["Amount"]])
X_test[["Amount"]] = scaler.transform(X_test[["Amount"]])

print("Before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(sampling_strategy=0.10, random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print(y_train_sm.value_counts())

## 4. Train Two Models

In [ ]:
log_reg = LogisticRegression(max_iter=300, solver="liblinear", random_state=42)
decision_tree = DecisionTreeClassifier(max_depth=12, class_weight="balanced", random_state=42)

log_reg.fit(X_train_sm, y_train_sm)
decision_tree.fit(X_train_sm, y_train_sm)

models = {
    "Logistic Regression": log_reg,
    "Decision Tree": decision_tree
}

## 5. Evaluation: Precision, Recall, F1 and ROC-AUC

In [ ]:
results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results)
results_df.round(4)

In [ ]:
# ROC-AUC curves
plt.figure(figsize=(8,6))
for name, model in models.items():
    prob = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
plt.plot([0,1],[0,1],"--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC-AUC Curve")
plt.legend()
plt.show()

## 6. Which Metric Matters Most?

**Recall is usually the most important metric in fraud detection** because a false negative means a fraudulent transaction was missed.

However, recall cannot be maximized blindly. If the model flags too many legitimate transactions, investigators or customers face many false alarms. Therefore, **Precision is also important**. The practical goal is high recall with acceptable precision, supported by F1-score and ROC-AUC.

In [ ]:
# Feature importance from Decision Tree
importance = pd.Series(decision_tree.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False).head(15)

plt.figure(figsize=(9,6))
importance.sort_values().plot(kind="barh")
plt.title("Top 15 Decision Tree Feature Importances")
plt.xlabel("Importance")
plt.show()

# Logistic Regression coefficients
coefficients = pd.Series(log_reg.coef_[0], index=X.columns)
print("Top positive coefficients:")
print(coefficients.sort_values(ascending=False).head(10))
print("\nTop negative coefficients:")
print(coefficients.sort_values().head(10))

## 7. Scalability: 1 Million Transactions per Hour

At 1 million transactions/hour, the model should be deployed as a real-time scoring service rather than run manually in Jupyter.

A scalable architecture would use:
- Kafka/message queues for streaming transactions.
- Parallel feature-processing workers.
- A lightweight model API for low-latency predictions.
- Batch/stream inference with autoscaling.
- Monitoring for data drift and fraud-rate changes.
- Periodic retraining using newly labelled fraud cases.
- Fast storage for predictions, alerts and investigation results.

Model inference is much cheaper than model training, so training can run asynchronously while prediction services remain available.

## 8. Conclusion

The dataset is extremely imbalanced, so accuracy alone would give a misleading picture. Stratified splitting keeps fraud cases in both train and test sets, while SMOTE improves minority-class representation **only in the training data**. Logistic Regression and Decision Tree are evaluated using Precision, Recall, F1-score and ROC-AUC.

For production fraud detection, recall is particularly important because missed fraud can directly create financial losses. Precision must still be monitored to control false alerts.